# Vector-like fermions in `feynlag`: SM + a vector-like lepton doublet

A **vector-like fermion** has both chiralities in the *same* gauge
representation, so — unlike a chiral SM fermion — it can carry a bare,
gauge-invariant Dirac mass all by itself [KM15]. Mixed with an ordinary SM fermion
through the Higgs, it produces a textbook flavor-mixing problem: a 2×2 mass
matrix diagonalized by *two independent* angles (θ_L, θ_R, one per
chirality), and — because the heavy and light states are no longer pure
gauge eigenstates — flavor-changing neutral currents (FCNCs) that don't
exist in the SM.

This notebook walks through `examples/sm_vll.py` stage by stage, reusing
the same declare → write → check → break-symmetry → diagonalize → extract
pipeline as **[`SM_Feynman_Rules_Tutorial.ipynb`](SM_Feynman_Rules_Tutorial.ipynb)**
(read that one first if `ExternalParameter`, `Dmu`, `Scalar`, or `Model`
are unfamiliar — this notebook won't re-explain them). What's new here:

- a **vector-like doublet** Ψ = (N, E), built as two `WeylFermion`s with
  identical reps (not a `DiracFermion` — see §2),
- a **bare mass term** `-M Ψ̄_L Ψ_R` that needs no scalar VEV to exist,
- **biunitary (2×2 SVD) diagonalization** with two physically distinct
  angles, via the analytic `diagonalize_svd_2x2` helper,
- **rotating fermion fields** into their mass basis and re-extracting
  Feynman rules — which exercises `expand_bilinear`, a piece of machinery
  this model is the first to need.

The headline physics we'll derive and verify symbolically:

1. the **Z-FCNC lives only in the right-handed current** (the left-handed
   Z coupling is exactly protected),
2. a clean **Higgs-coupling sum rule** for the deviation from `-m_e/v`,
3. a **new right-handed W current** with no SM analogue.


In [ ]:
import sympy as sp
from feynlag import (
    Bilinear, DiracGamma, FL_I, FL_J, Lagrangian, Model, ParameterSet, SU2,
    Scalar, U1, WeylFermion, bar_partner, diracPL, diracPR, dirac_mass,
    expand_bilinear, extract_fermion_vertices, fermion_mass_matrix, inner,
    kinetic, rotation_2x2, yukawa,
)

## 1. Symmetries and parameters

Same `SU2L`/`U1Y` electroweak setup as the plain-SM tutorial, plus three
new couplings for the VLL sector:

- `ye` — the ordinary (tiny) SM electron Yukawa,
- `lamE` — the mixing Yukawa between the heavy doublet and `e_R`,
- `MPsi` — the bare vector-like mass. It needs `unit_dim=1` (mass
  dimension 1) so `Model.check_invariance()`'s power-counting check can
  see that `MPsi · (dimension-3 Bilinear)` is a dimension-4 operator —
  exactly like `v` needs `unit_dim=1` for the Yukawa terms below.

Each parameter is declared once: `P.external(...)` / `P.internal(...)`
creates it **and** registers it in the `ParameterSet` the `Model` is given,
so no parameter list has to be kept in sync by hand.

In [ ]:
P = ParameterSet()
gw = P.external("gw", 0.6535, positive=True)
g1 = P.external("g1", 0.3580, positive=True)
SU2L, U1Y = SU2("SU2L", coupling=gw), U1("U1Y", coupling=g1)

v = P.external("v", 246.0, positive=True, unit_dim=1)
lam = P.external("lam", 0.129, tex="\\lambda")
mu2 = P.internal("mu2", unit_dim=2, tex="\\mu^2")

ye = P.external("ye", 0.01, positive=True, tex="y_e")
lamE = P.external("lamE", 0.4, positive=True, tex="\\lambda_E")
MPsi = P.external("MPsi", 1000.0, positive=True, unit_dim=1, tex="M_\\Psi")
MPsi.unit_dim

1

## 2. Fields

The Higgs and the SM lepton generation are declared exactly as in the SM
tutorial (one flavor: `nflavors=1`, so the mixing physics isn't obscured
by 3-generation flavor indices).


In [ ]:
H = Scalar("H", reps={SU2L: 2, U1Y: sp.Rational(1, 2)},
           component_names=["Gp", "H0"])
H.expand_vev({H.H0: v})

Ll = WeylFermion("Ll", reps={SU2L: 2, U1Y: -sp.Rational(1, 2)},
                 chirality="L", nflavors=1, component_names=["nuL", "eL"])
eRf = WeylFermion("eRf", reps={U1Y: -1}, chirality="R", nflavors=1,
                  component_names=["eR"])
Ll.components, eRf.components

([nuL, eL], [eR])

Now the vector-like doublet. `PsiL` and `PsiR` are given the **same**
reps `(SU2L: 2, U1Y: -1/2)` as `Ll` — both chiralities in one
representation is precisely what makes a bare mass gauge invariant (a
chiral SM fermion can't have one: `L_L` and `e_R` transform differently,
so nothing gauge-invariant connects them without a scalar).

Why two `WeylFermion`s and not a single `DiracFermion`? Because
`DiracFermion` is deliberately unsupported: it **raises on construction**.
A genuine four-component field would need its own mass and diagonalization
path, validated against physics, before it could be trusted. Two
`WeylFermion`s with identical reps (one `chirality='L'`, one `'R'`) reuse the
tested Weyl machinery at every stage, and that is exactly what a
vector-like fermion is (see `fields.py`'s `DiracFermion` docstring).


In [ ]:
PsiL = WeylFermion("PsiL", reps={SU2L: 2, U1Y: -sp.Rational(1, 2)},
                   chirality="L", nflavors=1, component_names=["NL", "EL"])
PsiR = WeylFermion("PsiR", reps={SU2L: 2, U1Y: -sp.Rational(1, 2)},
                   chirality="R", nflavors=1, component_names=["NR", "ER"])
W, B = SU2L.bosons("W"), U1Y.bosons("B")
PsiL.components, PsiR.components


([NL, EL], [NR, ER])

In [ ]:
# components by their declared names (bars: <name>bar).  FL_I/FL_J are the
# canonical flavour indices every term constructor uses, so the mass-basis
# rotations below reach every term with one substitution.
i, j = FL_I, FL_J
nuL, eL, nuLbar, eLbar = Ll.nuL, Ll.eL, Ll.nuLbar, Ll.eLbar
eR, eRbar = eRf.eR, eRf.eRbar
NL, EL, NLbar, ELbar = PsiL.NL, PsiL.EL, PsiL.NLbar, PsiL.ELbar
NR, ER, NRbar, ERbar = PsiR.NR, PsiR.ER, PsiR.NRbar, PsiR.ERbar

## 3. The Lagrangian

Three Yukawa-sector pieces, one constructor call each. A constructor
(`yukawa`, `dirac_mass`) does the gauge contraction, the flavour index and
the `+ h.c.`, and returns **the same component-level expression** you
would otherwise type by hand — each cell's output shows it
(`Bilinear(bar, gamma, field)` is the opaque fermion-sandwich atom;
`diracPR`/`diracPL` are the chiral projectors, and the `h.c.` half carries
`P_L` — see the SM tutorial for the full explanation of this convention).

### SM Yukawa: `-y_e L̄_L H e_R + h.c.`

Identical in form to the plain-SM lepton Yukawa.

In [ ]:
LYuk = yukawa(ye, Ll, H, eRf)       # −y_e L̄_L H e_R + h.c.
LYuk

-Gp*ye*Bilinear(nuLbar[fl_i], PR, eR[fl_i]) - H0*ye*Bilinear(eLbar[fl_i], PR, eR[fl_i]) - ye*conjugate(Gp)*Bilinear(eRbar[fl_i], PL, nuL[fl_i]) - ye*conjugate(H0)*Bilinear(eRbar[fl_i], PL, eL[fl_i])

### Mixing Yukawa: `-λ_E Ψ̄_L H e_R + h.c.`

`Ψ_L` carries exactly `L_L`'s reps, so the call and its component pattern
are identical to the SM Yukawa above — just with `NL`/`EL` in place of `nuL`/`eL`. This
single term is what ties the heavy doublet to the light SM lepton.

In [ ]:
LYuk += yukawa(lamE, PsiL, H, eRf)  # −λ_E Ψ̄_L H e_R + h.c.
LYuk

-Gp*lamE*Bilinear(NLbar[fl_i], PR, eR[fl_i]) - Gp*ye*Bilinear(nuLbar[fl_i], PR, eR[fl_i]) - H0*lamE*Bilinear(ELbar[fl_i], PR, eR[fl_i]) - H0*ye*Bilinear(eLbar[fl_i], PR, eR[fl_i]) - lamE*conjugate(Gp)*Bilinear(eRbar[fl_i], PL, NL[fl_i]) - lamE*conjugate(H0)*Bilinear(eRbar[fl_i], PL, EL[fl_i]) - ye*conjugate(Gp)*Bilinear(eRbar[fl_i], PL, nuL[fl_i]) - ye*conjugate(H0)*Bilinear(eRbar[fl_i], PL, eL[fl_i])

### Bare vector-like mass: `-M_Ψ Ψ̄_L Ψ_R + h.c.`

No Higgs field appears — this term needs no VEV to exist, which is the
whole point of "vector-like." Both `N` and `E` get this same mass `M_Ψ`.
(`dirac_mass` refuses two fields in different representations: a chiral
fermion's mass can only come from a Yukawa term.)

In [ ]:
LYuk += dirac_mass(MPsi, PsiL, PsiR)   # −M_Ψ Ψ̄_L Ψ_R + h.c.
LYuk

-Gp*lamE*Bilinear(NLbar[fl_i], PR, eR[fl_i]) - Gp*ye*Bilinear(nuLbar[fl_i], PR, eR[fl_i]) - H0*lamE*Bilinear(ELbar[fl_i], PR, eR[fl_i]) - H0*ye*Bilinear(eLbar[fl_i], PR, eR[fl_i]) - MPsi*Bilinear(ELbar[fl_i], PR, ER[fl_i]) - MPsi*Bilinear(ERbar[fl_i], PL, EL[fl_i]) - MPsi*Bilinear(NLbar[fl_i], PR, NR[fl_i]) - MPsi*Bilinear(NRbar[fl_i], PL, NL[fl_i]) - lamE*conjugate(Gp)*Bilinear(eRbar[fl_i], PL, NL[fl_i]) - lamE*conjugate(H0)*Bilinear(eRbar[fl_i], PL, EL[fl_i]) - ye*conjugate(Gp)*Bilinear(eRbar[fl_i], PL, nuL[fl_i]) - ye*conjugate(H0)*Bilinear(eRbar[fl_i], PL, eL[fl_i])

Finally the gauge currents. `kinetic(...)` returns the canonical
kinetic/gauge terms of a list of fields — `(D_μφ)†(D^μφ)` for a scalar, the
gauge current for a fermion (one generic `fermion_gauge_current` each,
regardless of how many gauge groups it is charged under). Below,
`L.add_kinetic(...)` adds exactly these; here are the four lepton currents:

In [ ]:
current = sp.Add(*[expr for expr, _sector in kinetic(Ll, eRf, PsiL, PsiR)])
current

-B*g1*Bilinear(ELbar[fl_i], gamma(mu)*PL, EL[fl_i])/2 - B*g1*Bilinear(ERbar[fl_i], gamma(mu)*PR, ER[fl_i])/2 - B*g1*Bilinear(NLbar[fl_i], gamma(mu)*PL, NL[fl_i])/2 - B*g1*Bilinear(NRbar[fl_i], gamma(mu)*PR, NR[fl_i])/2 - B*g1*Bilinear(eLbar[fl_i], gamma(mu)*PL, eL[fl_i])/2 - B*g1*Bilinear(eRbar[fl_i], gamma(mu)*PR, eR[fl_i]) - B*g1*Bilinear(nuLbar[fl_i], gamma(mu)*PL, nuL[fl_i])/2 + W_1*gw*Bilinear(ELbar[fl_i], gamma(mu)*PL, NL[fl_i])/2 + W_1*gw*Bilinear(ERbar[fl_i], gamma(mu)*PR, NR[fl_i])/2 + W_1*gw*Bilinear(NLbar[fl_i], gamma(mu)*PL, EL[fl_i])/2 + W_1*gw*Bilinear(NRbar[fl_i], gamma(mu)*PR, ER[fl_i])/2 + W_1*gw*Bilinear(eLbar[fl_i], gamma(mu)*PL, nuL[fl_i])/2 + W_1*gw*Bilinear(nuLbar[fl_i], gamma(mu)*PL, eL[fl_i])/2 + I*W_2*gw*Bilinear(ELbar[fl_i], gamma(mu)*PL, NL[fl_i])/2 + I*W_2*gw*Bilinear(ERbar[fl_i], gamma(mu)*PR, NR[fl_i])/2 - I*W_2*gw*Bilinear(NLbar[fl_i], gamma(mu)*PL, EL[fl_i])/2 - I*W_2*gw*Bilinear(NRbar[fl_i], gamma(mu)*PR, ER[fl_i])/2 + I*W_2*gw*Bilinear(eLbar[fl_i], gam

## 4. Assemble the Model and check invariance

The kinetic/potential Higgs sector is the standard SM piece. Assembling
everything into a `Model` and calling `check_invariance()` is the
correctness gate: this is the first model in `feynlag` where a bare
fermion mass *and* a mixing Yukawa **between two different `Fermion`
objects** get checked for gauge invariance, hermiticity (via
`Bilinear._eval_conjugate` and the `bar_partner` registry — which has to
correctly pair `Ll`/`eRf` legs with `PsiL`/`PsiR` legs), and mass
dimension all at once. `model.summary()` prints what was declared: every
field with its representations, and every parameter.

In [ ]:
HdH = inner(H, H)                      # H†H
V = -mu2 * HdH + lam * HdH**2

L = Lagrangian()
L.add_kinetic(H, Ll, eRf, PsiL, PsiR)  # (D_μH)†(D^μH) + the gauge currents
L.add(-V, sector="potential")
L.add(LYuk, sector="yukawa")

model = Model("SM-VLL", gauge_groups=[SU2L, U1Y],
              fields=[H, Ll, eRf, PsiL, PsiR, W, B],
              parameters=P, lagrangian=L)
print(model.summary())

report = model.check_invariance()
report.raise_on_failure()
report

Model: SM-VLL

Fields
field  type       SU2L  U1Y   flavours  components     VEV
-----  ---------  ----  ----  --------  -------------  -----------
H      scalar     2     1/2             Gp, H0         ⟨H0⟩ = v/√2
Ll     fermion L  2     -1/2  1         nuL, eL
eRf    fermion R  1     -1    1         eR
PsiL   fermion L  2     -1/2  1         NL, EL
PsiR   fermion R  2     -1/2  1         NR, ER
W      vector     adj   0               W_1, W_2, W_3
B      vector     1     0               B

Parameters
parameter  nature    value / expression  mass dim
---------  --------  ------------------  --------
gw         external  0.6535              0
g1         external  0.358               0
v          external  246.0               1
lam        external  0.129               0
mu2        internal                      2
ye         external  0.01                0
lamE       external  0.4                 0
MPsi       external  1000.0              1


InvarianceReport(24 checks, OK)

## 5. EWSB and the mass blocks

Solve the tadpole for `mu2`. Then let the model find its own **mass
blocks**: which fluctuations mix, each with its electric charge (derived
from the vacuum, never typed as `Q = T3 + Y`) and its Goldstone
directions. The gauge blocks are rotated right away — `(W1, W2) -> (W+, W-)`
and `(W3, B) -> (Z, A)`, the same rotations as the SM helpers
`charged_current_rotation`/`weinberg_rotation`. The charged-lepton block
`[eL, EL] x [eR, ER]` is the subject of §6–§8.

In [ ]:
model.solve_tadpoles([mu2])


{mu2: lam*v**2}

In [ ]:
for block in model.mass_blocks():
    print(block)

charged, neutral = model.mass_blocks(kind="gauge")
# names: massive states first, the photon last; for the charged block the
# positive-charge states and their conjugates
Z, A = model.diagonalize(neutral, names=["Z", "A"]).new_fields
Wp, Wm = model.diagonalize(charged, names=["W^+"],
                           conjugate_names=["W^-"]).rotation.new_fields

scalar  Q=0  cp-even  [H0_r]  goldstones: 0
scalar  Q=0  cp-odd  [H0_i]  goldstones: 1
scalar  Q=1  [Gp]  goldstones: 1
gauge  Q=1  [W_1, W_2]  massless: 0
gauge  Q=0  [W_3, B]  massless: 1
fermion  Q=0  [nuL]  x []  (massless: no Dirac partner)
fermion  Q=-1  [eL, EL]  x [eR, ER]
fermion  Q=0  [NL]  x [NR]


## 6. The charged-lepton mass matrix

`fermion_mass_matrix(L, bar_base, field_base, ...)` extracts the
coefficient of one `(bar, field)` `Bilinear` pair from the vacuum-shifted
Lagrangian — it makes no assumption that `bar_base` and `field_base`
belong to the same `Fermion` object, so calling it 4× over
`(ē_L, Ē_L) × (e_R, E_R)` and assembling a 2×2 by hand gives the full
mixing mass matrix:


In [ ]:
bars, rights = (eLbar, ELbar), (eR, ER)
M2 = sp.Matrix(2, 2, lambda a, b: fermion_mass_matrix(
    LYuk, bars[a], rights[b], model.vacuum, 1, (i, j), gamma=diracPR)[0, 0])
M2

Matrix([
[  sqrt(2)*v*ye/2,    0],
[sqrt(2)*lamE*v/2, MPsi]])

The lower-triangular shape (zero in the top-right) says something
physical: there is no bare `L̄_L Ψ_R` mass in our Lagrangian (it's
removable by a field redefinition, so we never wrote one) — the only way
`e_R` talks to the doublet sector is through `λ_E`.

The neutral sector has no such mixing at all: `N` is a pure Dirac state
of mass exactly `M_Ψ`, and the SM neutrino stays exactly massless (no
`Bilinear` anywhere pairs `nuL` with a right-handed state).


In [ ]:
m_N = fermion_mass_matrix(LYuk, NLbar, NR, model.vacuum, 1, (i, j),
                          gamma=diracPR)[0, 0]
m_nu = fermion_mass_matrix(LYuk, nuLbar, NR, model.vacuum, 1, (i, j),
                           gamma=diracPR)[0, 0]
m_N, m_nu


(MPsi, 0)

## 7. Biunitary diagonalization

A non-symmetric mass matrix needs *two* rotations — one for the
bar/left-handed fields, one for the field/right-handed ones — the
biunitary (SVD) construction: `θ_L` diagonalizes `M·Mᵀ`, `θ_R`
diagonalizes `Mᵀ·M`. `diagonalize_svd`'s generic route
(`Matrix.diagonalize(normalize=True)`) returns unusable nested-radical
expressions for symbolic entries, so `model.diagonalize` on a symbolic 2×2
Dirac block instead calls the same analytic `tan 2θ` machinery used for
scalar mixing (`diagonalize_svd_2x2` → `solve_mixing_angle_2x2`) on `M·Mᵀ`
and `Mᵀ·M` separately. `angle=(thL, thR)` names the two angles and
`names=["e1", "e2"]` the mass states, which become the `IndexedBase`s
`e1L, e1R, e1Lbar, e1Rbar, …`.

Given the matrix's shape above (`M[0,1]=0`), expect `θ_L` to be small —
it can only arise from the *product* `y_e λ_E`, both small numbers —
while `θ_R` picks up the direct `λ_E v / M_Ψ` mixing and is the
physically large angle. That's the defining signature of a vector-like
**doublet** (for a vector-like **singlet** it's the other way around).

In [ ]:
(block,) = [b for b in model.mass_blocks(kind="fermion") if eL in b.fields]
assert block.matrix == M2          # the matrix extracted by hand above

thL, thR = sp.symbols("thL thR", real=True)
d = model.diagonalize(block, names=["e1", "e2"], angle=(thL, thR))
rotL, rotR = d.rotations[0], d.rotations[2]
e1L, e2L = d.new_fields
e1R, e2R = d.new_right
e1Lbar, e2Lbar = (bar_partner(f) for f in d.new_fields)
e1Rbar, e2Rbar = (bar_partner(f) for f in d.new_right)
rotL.angle_relation, rotR.angle_relation

(Eq(tan(2*thL), -2*lamE*v**2*ye/(2*MPsi**2 + lamE**2*v**2 - v**2*ye**2)),
 Eq(2*tan(thR)/(1 - tan(thR)**2), (2*sqrt(2)*MPsi*lamE*v*sin(atan(2*lamE*v**2*ye/(2*MPsi**2 + lamE**2*v**2 - v**2*ye**2))/2)**2 - 2*sqrt(2)*MPsi*v*ye*sin(atan(2*lamE*v**2*ye/(2*MPsi**2 + lamE**2*v**2 - v**2*ye**2))/2)*cos(atan(2*lamE*v**2*ye/(2*MPsi**2 + lamE**2*v**2 - v**2*ye**2))/2))/(-2*MPsi**2*sin(atan(2*lamE*v**2*ye/(2*MPsi**2 + lamE**2*v**2 - v**2*ye**2))/2)**2 + lamE**2*v**2*sin(atan(2*lamE*v**2*ye/(2*MPsi**2 + lamE**2*v**2 - v**2*ye**2))/2)**2 - 2*lamE*v**2*ye*sin(atan(2*lamE*v**2*ye/(2*MPsi**2 + lamE**2*v**2 - v**2*ye**2))/2)*cos(atan(2*lamE*v**2*ye/(2*MPsi**2 + lamE**2*v**2 - v**2*ye**2))/2) + v**2*ye**2*cos(atan(2*lamE*v**2*ye/(2*MPsi**2 + lamE**2*v**2 - v**2*ye**2))/2)**2)))

Dual verification (symbolic derivation + numeric check, per
`CONVENTIONS.md`): plug in a benchmark point and confirm the rotation
really diagonalizes `M`, with the light state landing in slot 1.


In [ ]:
bench = {ye.s: 0.01, lamE.s: 0.4, v.s: 246.0, MPsi.s: 1000.0}
thL_num = float(rotL.angle_solution.subs(bench))
thR_num = float(rotR.angle_solution.subs(bench))
D = rotation_2x2(thL_num) * M2.subs(bench) * rotation_2x2(thR_num).T

assert abs(D[0, 1]) < 1e-9 and abs(D[1, 0]) < 1e-9
assert abs(D[0, 0]) < abs(D[1, 1])   # field 1 = light state
thL_num, thR_num, float(D[0, 0]), float(D[1, 1])


(-0.00012044923364816905,
 -0.06946755651195813,
 1.735287235094696,
 1002.417724593002)

## 8. What `diagonalize` registered

Rewriting the Lagrangian in the mass basis uses the *same* `Rotation`
machinery as the boson case above — `Rotation([eL[i], EL[i]], [e1L[i],
e2L[i]], rotation_2x2(θ))` — but with `Indexed` old-fields instead of
plain `Symbol`s. Two things are specific to fermions, and `diagonalize`
takes care of both:

1. **Two rotations per chirality**, not one: the field-side rotation
   (`eL, EL -> e1L, e2L`) and an *independent* bar-side rotation
   (`eLbar, ELbar -> e1Lbar, e2Lbar`) with the same matrix — a `Bilinear`
   has a bar leg and a field leg, and both need rewriting.
2. **A shared flavor-index symbol.** Every term above was built with the
   same `FL_I`, so a single `xreplace` reaches every occurrence at once.

Here's the subtlety underneath. After `xreplace`, a `Bilinear` slot holds
`cosθ·e1L[i] + sinθ·e2L[i]` — a SymPy `Add` — trapped inside an opaque,
custom `Function` that has no linearity rule built in. Watch what a raw
`Bilinear` looks like right after the substitution, before anything
distributes it:

In [ ]:
demo_bar, demo_field = sp.IndexedBase("demo_bar"), sp.IndexedBase("demo_field")
demo_new1, demo_new2 = sp.IndexedBase("demo_n1"), sp.IndexedBase("demo_n2")
th_demo = sp.Symbol("th_{demo}", real=True)

demo_term = Bilinear(demo_bar[i], diracPR, demo_field[i])
# what Rotation.substitution() would build for field[i] -> R^-1 * new[i];
# written out directly here since a real Rotation needs 2 old fields:
demo_sub = {demo_field[i]: sp.cos(th_demo) * demo_new1[i]
                          + sp.sin(th_demo) * demo_new2[i]}
demo_rotated = demo_term.xreplace(demo_sub)
demo_rotated, demo_rotated.field


(Bilinear(demo_bar[fl_i], PR, sin(th_{demo})*demo_n2[fl_i] + cos(th_{demo})*demo_n1[fl_i]),
 sin(th_{demo})*demo_n2[fl_i] + cos(th_{demo})*demo_n1[fl_i])

The `field` slot is now an unresolved `Add` — `extract_fermion_vertices`
would group by this *entire* composite key rather than splitting it into
two clean `(coefficient, Indexed)` contributions. `expand_bilinear`
(added to `vertices/bilinear.py` alongside this example) distributes
`Bilinear` over `Add`-valued legs, exploiting that it's linear in each of
its bar/field slots:


In [ ]:
expand_bilinear(demo_rotated)


sin(th_{demo})*Bilinear(demo_bar[fl_i], PR, demo_n2[fl_i]) + cos(th_{demo})*Bilinear(demo_bar[fl_i], PR, demo_n1[fl_i])

Two clean terms, each with a plain `Indexed` leg and the right
`cosθ`/`sinθ` coefficient. `extract_fermion_vertices` and
`fermion_mass_matrix` both call `expand_bilinear` automatically now, so
the real model below doesn't need to do this by hand — but this is
exactly what would silently go wrong without it.

The four rotations `diagonalize` registered on the model (old fields →
new fields):

In [ ]:
[(r.old_fields, r.new_fields) for r in d.rotations]

[([eL[fl_i], EL[fl_i]], [e1L[fl_i], e2L[fl_i]]),
 ([eLbar[fl_i], ELbar[fl_i]], [e1Lbar[fl_i], e2Lbar[fl_i]]),
 ([eR[fl_i], ER[fl_i]], [e1R[fl_i], e2R[fl_i]]),
 ([eRbar[fl_i], ERbar[fl_i]], [e1Rbar[fl_i], e2Rbar[fl_i]])]

## 9. Extracting the physical couplings

`physical_lagrangian` applies the vacuum shift, tadpole solution, and
every registered rotation (bosonic *and* fermionic) in one pass;
`extract_fermion_vertices` then groups by `(bar, gamma, field)` and peels
off the boson legs — exactly as in the SM tutorial.


In [ ]:
h = H.H0_r                 # the Higgs fluctuation, by name
LYuk_phys = model.physical_lagrangian(sector="yukawa")
table = extract_fermion_vertices(LYuk_phys, [Z, A, Wp, Wm, h])

mu = sp.Symbol("mu", integer=True)
gammaL = DiracGamma(mu) * diracPL
gammaR = DiracGamma(mu) * diracPR


def coeff(bar, gamma, field, boson):
    entry = table.get((bar, gamma, field))
    if entry is None:
        return sp.S.Zero
    return entry.get(1, {}).get((boson,), sp.S.Zero)

### Z couplings — the headline result

`e_L` and `E_L` are **both** `T³ = -1/2` doublet members, so rotating
between them by `θ_L` doesn't change the diagonal Z coupling and produces
**exactly zero** FCNC — the left-handed current is protected by the
doublet structure, independent of any mixing angle.

`e_R` (`T³ = 0`, an SM singlet) and `E_R` (`T³ = -1/2`, from the doublet)
are *different* under `SU(2)_L`, so rotating them by `θ_R` both shifts
the light state's diagonal coupling **and** generates a genuine FCNC.


In [ ]:
sp.simplify(coeff(e1Lbar[i], gammaL, e1L[i], Z))   # LH diagonal: unchanged


(g1**2 - gw**2)/(2*sqrt(g1**2 + gw**2))

In [ ]:
sp.simplify(coeff(e1Lbar[i], gammaL, e2L[i], Z))   # LH FCNC: exactly zero


0

In [ ]:
sp.simplify(coeff(e1Rbar[i], gammaR, e1R[i], Z))   # RH diagonal: shifted


(-g1**2*sin(thR)**2 + 2*g1**2 - gw**2*sin(thR)**2)/(2*sqrt(g1**2 + gw**2))

In [ ]:
sp.simplify(coeff(e1Rbar[i], gammaR, e2R[i], Z))   # RH FCNC: -(gZ/2) sinθR cosθR


-sqrt(g1**2 + gw**2)*sin(2*thR)/4

### Photon couplings — a sanity check

`U(1)_EM` is unbroken, so the rotation must not touch the electric
charge: diagonal coupling `-e` in both mass eigenstates, and **zero**
photon FCNC (a mismatch here would mean a bug in the rotation, not new
physics).


In [ ]:
sp.simplify(coeff(e1Lbar[i], gammaL, e1L[i], A))

-g1*gw/sqrt(g1**2 + gw**2)

In [ ]:
sp.simplify(coeff(e1Lbar[i], gammaL, e2L[i], A))

0

### W couplings — a genuinely new right-handed current

The left-handed current splits between the light and heavy states by the
usual `cosθ_L`/`sinθ_L` pattern. But because `Ψ_R` is *also* an `SU(2)_L`
doublet (unlike the SM's `e_R` singlet), there's a **right-handed** charged
current `W⁺ N̄_R e_R ∝ sinθ_R` that has no SM analogue at all.


In [ ]:
sp.simplify(coeff(nuLbar[i], gammaL, e1L[i], Wp))


sqrt(2)*gw*cos(thL)/2

In [ ]:
sp.simplify(coeff(nuLbar[i], gammaL, e2L[i], Wp))

-sqrt(2)*gw*sin(thL)/2

In [ ]:
sp.simplify(coeff(NRbar[i], gammaR, e1R[i], Wp))   # the new RH current


sqrt(2)*gw*sin(thR)/2

### Higgs couplings — a sum rule

The light lepton's Higgs coupling deviates from the SM value `-m_e/v`
by a clean, closed-form amount: `sinθ_L sinθ_R M_Ψ/v`. Rather than just
printing the coupling, build the SM mass formula for this mixed system
and verify the deviation symbolically — the residue below should
`simplify` to exactly zero.


In [ ]:
cL, sL = sp.cos(thL), sp.sin(thL)
cR, sR = sp.cos(thR), sp.sin(thR)
m_e_expr = (cL * ye.s + sL * lamE.s) * v.s / sp.sqrt(2) * cR + sL * sR * MPsi.s

h_e1e1 = coeff(e1Lbar[i], diracPR, e1R[i], h)
residue = sp.simplify(h_e1e1 + (m_e_expr - sL * sR * MPsi.s) / v.s)
residue


0

## 10. Recap

Starting from one bare mass term and one mixing Yukawa, `feynlag`
mechanically derived: a non-trivial 2×2 mixing matrix, two independent
diagonalization angles with the doublet's characteristic
θ_L-small/θ_R-large hierarchy, an exactly-zero left-handed Z-FCNC
alongside a nonzero right-handed one, a new right-handed W current, and a
closed-form Higgs-coupling sum rule — all *derived*, not assumed, and all
cross-checked symbolically above.

- Exact pinned formulas for every coupling here (including the ones we
  only spot-checked): [`tests/test_vll.py`](../../tests/test_vll.py).
- Every step above used the field-level constructors and the guided
  physical basis; the component-level forms they return are pinned in
  [`tests/test_terms.py`](../../tests/test_terms.py) and
  [`tests/test_physical_basis.py`](../../tests/test_physical_basis.py).
- The full flat script this notebook walks through:
  [`examples/sm_vll.py`](../../examples/sm_vll.py).
- For the plain-SM baseline this model extends:
  [`SM_Feynman_Rules_Tutorial.ipynb`](SM_Feynman_Rules_Tutorial.ipynb) /
  [`examples/sm_scalar_gauge.py`](../../examples/sm_scalar_gauge.py).

---
## 11. References

- **[KM15]** N. Kumar, S. P. Martin, *Vectorlike leptons at the Large Hadron Collider*,
  Phys. Rev. D **92** (2015) 115018,
  doi:[10.1103/PhysRevD.92.115018](https://doi.org/10.1103/PhysRevD.92.115018),
  [arXiv:1510.03456](https://arxiv.org/abs/1510.03456) — the doublet and singlet
  vector-like lepton models, their mixing with the SM leptons and their modified
  $W$/$Z$/$h$ couplings.